# AutoML über den Aufbau: kombiniert, hierarchisch, getrennt

**Autor:** jd · **Datum:** 2026-10-28 · **Issue:** –

**Ziel:** Die erste Optuna-Suche (`12_jd_automl-optuna`) hat nur die kombinierte Klasse
durchsucht. Hier wählt die Suche zusätzlich den **Aufbau**:

| Aufbau | Modell | Besonderheit |
| --- | --- | --- |
| kombiniert | `CombinedLabelClassifier` | eine Klasse je `Crop|Stage`-Paar |
| hierarchisch | `HierarchicalClassifier` | Kultur, dann Stadium je Kultur; jede Stufe wählt ihre eigene Modellfamilie |
| getrennt | `MultiOutputClassifier` | zwei unabhängige Modelle, kann unmögliche Paare vorhersagen |

Modellfamilien: SVM, Extra Trees, MLP (HistGradientBoosting und logistische Regression lagen in der
ersten Suche klar zurück). Referenz: `combined_svm_aez_month` (CV 0.864, Validierung 0.869).

**Aufbau:** Teil A erzeugt die Läufe (MLflow), Teil B belegt jede Erkenntnis mit einem Plot.

In [ ]:
%load_ext autoreload
%autoreload 2

from typing import get_args

import pandas as pd
from optuna.visualization.matplotlib import plot_optimization_history, plot_param_importances

from awp2.automl import AutoMLConfig, automl
from awp2.config import (
    AUTOML_APPROACH,
    COMBINED_STRUCTURE,
    EXTRA_TREES_FAMILY,
    MLP_FAMILY,
    MODEL_DOCS_FIGURES_DIR,
    SVM_FAMILY,
    ModelStructure,
)
from awp2.data import load_split
from awp2.evaluation import compare_predictions, plot_confusion_matrices
from awp2.experiment import RunConfig, out_of_fold_predictions, run
from awp2.models.combined import make_combined_svm
from awp2.plots import (
    plot_class_recall,
    plot_cv_scores,
    plot_metric_comparison,
    plot_top_confusions,
    save_doc_figure,
)
from awp2.preprocessing import PreprocessingConfig

# A · Läufe

### Referenz: bestes Modell bisher

In [ ]:
reference_model = make_combined_svm().set_params(estimator__C=300.0, estimator__gamma=0.003)
reference_config = RunConfig(
    name="combined_svm_aez_month",
    approach="combined",
    preprocessing=PreprocessingConfig(use_spectral_standard_scale=True, use_aez_month=True),
    track=False,
)
REFERENCE_CV = 0.864

### Suche

Höchstens 300 Versuche oder 1 h, Stand in `models/automl.db` (Fortsetzen mit demselben Namen).

In [ ]:
search = automl(
    AutoMLConfig(
        name="automl_structures",
        description="Optuna (TPE + Median-Pruning) über den Aufbau (kombiniert, hierarchisch mit "
        "eigener Familie je Stufe, getrennt), SVM/Extra Trees/MLP und das Preprocessing.",
        families=(SVM_FAMILY, EXTRA_TREES_FAMILY, MLP_FAMILY),
        structures=get_args(ModelStructure),
    )
)
print(f"Bester CV-Score: {search.best_score:.4f}")
search.best_params

### Gepaarter Vergleich auf dem Trainingsteil (out-of-fold)

In [ ]:
best_config = RunConfig(
    name="automl_structures_best",
    approach=AUTOML_APPROACH,
    preprocessing=search.best.preprocessing,
    balance_samples=search.best.balance_samples,
    track=False,
)
y_train = load_split().y_train
oof_reference = out_of_fold_predictions(reference_model, reference_config)
oof_best = out_of_fold_predictions(search.best.model, best_config)
compare_predictions(y_train, oof_reference, oof_best)

### Bester Kandidat: einmal auf der Validierung bewerten

In [ ]:
best = run(
    search.best.model,
    RunConfig(
        name="automl_structures_best",
        approach=AUTOML_APPROACH,
        description="Bester Versuch von automl_structures, einmal auf der Validierung.",
        preprocessing=search.best.preprocessing,
        balance_samples=search.best.balance_samples,
        tuning_run=search.run_id,
    ),
)
reference = run(reference_model, reference_config)
pd.DataFrame({"Referenz-SVM": reference.metrics.model_dump(), "AutoML": best.metrics.model_dump()})

# B · Erkenntnisse

## 1. Bester Versuch je Aufbau

In [ ]:
params = search.candidates["params"]
structures = params.map(lambda p: p.get("structure", COMBINED_STRUCTURE))
ax = plot_cv_scores(dict(tuple(search.candidates.groupby(structures))), reference=REFERENCE_CV)
ax.set_title("Optuna: bester CV-Score je Aufbau (gestrichelt: Referenz-SVM)")
save_doc_figure(ax.figure, "automl_aufbau", MODEL_DOCS_FIGURES_DIR)

## 2. Bester Versuch je Aufbau und Modellfamilie

Hierarchisch: Familie der Kultur-Stufe → Familie der Stadium-Stufe.

In [ ]:
def combination(p: dict) -> str:
    if "crop_family" in p:
        return f"hierarchisch: {p['crop_family']} → {p['stage_family']}"
    return f"{p.get('structure', COMBINED_STRUCTURE)}: {p['family']}"


ax = plot_cv_scores(
    dict(tuple(search.candidates.groupby(params.map(combination)))), reference=REFERENCE_CV
)
ax.set_title("Optuna: bester CV-Score je Kombination")
save_doc_figure(ax.figure, "automl_aufbau_familien", MODEL_DOCS_FIGURES_DIR)

## 3. Suchverlauf

In [ ]:
states = pd.Series([trial.state.name for trial in search.study.trials]).value_counts()
print(states.to_string())
print(structures.value_counts().to_string())
ax = plot_optimization_history(search.study)
ax.axhline(REFERENCE_CV, color="black", linestyle="--", linewidth=1, label="Referenz-SVM")
ax.legend()
save_doc_figure(ax.figure, "automl_aufbau_verlauf", MODEL_DOCS_FIGURES_DIR)

## 4. Welche Entscheidungen zählen

In [ ]:
ax = plot_param_importances(search.study)
save_doc_figure(ax.figure, "automl_aufbau_wichtigkeit", MODEL_DOCS_FIGURES_DIR)

## 5. Validierung: AutoML gegen die Referenz

In [ ]:
ax = plot_metric_comparison({"Referenz-SVM": reference.metrics, "AutoML (Aufbau)": best.metrics})
ax.set_title("AutoML über den Aufbau vs. Referenz-SVM")
save_doc_figure(ax.figure, "automl_aufbau_vergleich", MODEL_DOCS_FIGURES_DIR)

## 6. Fehlerbild des besten Kandidaten

In [ ]:
fig = plot_confusion_matrices(best.y_val, best.y_pred)
save_doc_figure(fig, "automl_aufbau_confusion", MODEL_DOCS_FIGURES_DIR)

In [ ]:
plot_top_confusions(best.y_val, best.y_pred)

In [ ]:
plot_class_recall(best.y_val, best.y_pred)

# Fazit

| | CV | BAcc Kultur | BAcc Stadium | BAcc kombiniert | Macro-F1 kombiniert |
| --- | --- | --- | --- | --- | --- |
| Referenz-SVM (`combined_svm_aez_month`) | 0.864 | 0.923 | 0.922 | 0.869 | 0.859 |
| AutoML (Aufbau), kombiniert + Extra Trees | 0.861 | 0.928 | 0.928 | **0.870** | 0.861 |

- Bester je Aufbau: kombiniert 0.861, hierarchisch 0.841 (Extra Trees → Extra Trees), getrennt
  0.837 (MLP) – dieselbe Reihenfolge wie in den Einzel-Experimenten.
- Einschränkung: TPE hat nur 6 hierarchische und 5 getrennte Versuche abgeschlossen; ein fairer
  Vergleich der Aufbauten bräuchte ein eigenes Budget je Aufbau.
- Gleichauf mit der Referenz (out-of-fold 170 : 156, p = 0.47) – kein neuer Bestwert.

Details: `docs/modelle/ansaetze/automl.md`.